# TouristGuide: LLM Agent with Tool-Calling
**Project 3 — Final Version**

---

### Architecture (5 layers)

```
User types a string
        │
        ▼
┌── [0] Interpretation Layer ─────────────────────────────┐
│   extract_city · extract_budget · extract_date          │
│   map_budget_level · extract_days · extract_trip_type   │
│   → UserIntent (structured, typed)                      │
└─────────────────────────────────────────────────────────┘
        │
        ▼
┌── [1] Planner ──────────────────────────────────────────┐
│   Phase 1: completeness check                           │
│   Phase 2: rule-based tool selector                     │
│   → Plan(tools, reasoning, requires_clarification)      │
└─────────────────────────────────────────────────────────┘
        │ plan.reasoning logged
        ▼
┌── [2] Tool Executor ────────────────────────────────────┐
│   Weather · Currency · Attractions                      │
│   Each tool returns typed schema {status, data, meta}   │
│   3 failure types: hard · soft · logical inconsistency  │
└─────────────────────────────────────────────────────────┘
        │
        ▼
┌── [3] Validator ────────────────────────────────────────┐
│   Normalise schemas · emit decision constraints         │
│   Compute GlobalStatus · Confidence score               │
│   → ValidationEnvelope                                  │
└─────────────────────────────────────────────────────────┘
        │ constraints BINDING from here
        ▼
┌── [4] Responder ────────────────────────────────────────┐
│   Receives ValidationEnvelope ONLY                      │
│   Never sees raw user string                            │
│   Confidence drives tone · Constraints enforced         │
└─────────────────────────────────────────────────────────┘
        │
        ▼
   Final output
```

### Design principles
- **Explicit state** — no bare `None`, no swallowed exceptions, no hidden assumptions
- **Separation of concerns** — each layer has exactly one job
- **Deterministic control** — rule-based routing, typed enums, no LLM in the planning path
- **Grounded output** — Responder only sees validated tool data, never the raw user message


---
## Section 1 — Data Contracts

All shared enums and dataclasses. Every tool returns `{status, data, meta}` — the anti-hallucination backbone.

In [ ]:
from dataclasses import dataclass, field
from enum import Enum
from typing import Optional
from datetime import date as Date

# ── Shared enums ──────────────────────────────────────────────────

class ToolStatus(str, Enum):
    SUCCESS     = "success"
    UNAVAILABLE = "unavailable"   # network / API down
    ERROR       = "error"         # bad input — unsupported city, invalid key
    EMPTY       = "empty"         # call succeeded but returned nothing useful

class Confidence(str, Enum):
    HIGH   = "high"
    MEDIUM = "medium"
    LOW    = "low"

class DataSource(str, Enum):
    API      = "api"
    FALLBACK = "fallback"
    CACHED   = "cached"

class GlobalStatus(str, Enum):
    COMPLETE     = "complete"      # all tools succeeded
    PARTIAL      = "partial"       # some tools failed — plan still possible
    INSUFFICIENT = "insufficient"  # critical data missing — plan heavily degraded
    TERMINAL     = "terminal"      # nothing to work with — abort

class ValidationIssue(str, Enum):
    WEATHER_UNAVAILABLE = "weather_unavailable"
    NO_ATTRACTIONS      = "no_attractions"
    BUDGET_MISSING      = "budget_missing"
    CURRENCY_FAILED     = "currency_failed"
    COST_UNKNOWN        = "cost_unknown"
    LOGICAL_CONFLICT    = "logical_conflict"

class DecisionConstraint(str, Enum):
    WEATHER_AWARE    = "weather_aware"
    WEATHER_AGNOSTIC = "weather_agnostic"

class RateType(str, Enum):
    LIVE   = "live"
    CACHED = "cached"
    FIXED  = "fixed"

class ActivityType(str, Enum):
    OUTDOOR  = "outdoor"
    INDOOR   = "indoor"
    CULTURAL = "cultural"
    MIXED    = "mixed"

class WeatherCondition(str, Enum):
    SUNNY         = "sunny"
    PARTLY_CLOUDY = "partly_cloudy"
    CLOUDY        = "cloudy"
    RAINY         = "rainy"
    THUNDERSTORM  = "thunderstorm"
    FOGGY         = "foggy"
    UNKNOWN       = "unknown"

class WeatherSuitability(str, Enum):
    GOOD    = "good"
    BAD     = "bad"
    NEUTRAL = "neutral"

SUPPORTED_CURRENCIES = {"USD", "EUR", "GBP", "UGX", "KES", "RWF", "TZS"}


# ── Layer 0: UserIntent ───────────────────────────────────────────
# Richer than TripParams — carries parsed NLP fields

@dataclass
class UserIntent:
    destination:   Optional[str]   = None
    start_date:    Optional[str]   = None   # YYYY-MM-DD after normalisation
    num_days:      Optional[int]   = None
    budget:        Optional[float] = None   # numeric (may be derived from budget_level)
    budget_level:  Optional[str]   = None   # "low" | "medium" | "high" — from NLP
    currency:      Optional[str]   = None
    trip_type:     Optional[str]   = None   # "leisure" | "adventure" | etc
    budget_was_inferred: bool      = False  # True if derived from budget_level


# ── Layer 1: Plan ─────────────────────────────────────────────────

@dataclass
class Plan:
    """Explicit plan — tools + reasoning trace. Every decision is auditable."""
    requires_clarification: bool
    missing_fields: list
    tools: list
    reasoning: str


# ── Layer 2: Tool output schemas ──────────────────────────────────

@dataclass
class WeatherData:
    temperature_celsius:        Optional[float]       = None
    condition:                  WeatherCondition       = WeatherCondition.UNKNOWN
    precipitation_probability:  Optional[float]        = None
    wind_speed_kph:             Optional[float]        = None

@dataclass
class WeatherMeta:
    source:               DataSource
    timestamp:            str
    confidence:           Confidence
    decision_constraint:  DecisionConstraint

@dataclass
class WeatherOutput:
    status: ToolStatus
    data:   WeatherData
    meta:   WeatherMeta

@dataclass
class CurrencyData:
    amount_input:     float
    currency_from:    str
    currency_to:      str
    converted_amount: Optional[float] = None
    exchange_rate:    Optional[float] = None
    input_validated:  bool            = False

@dataclass
class CurrencyMeta:
    source:     DataSource
    timestamp:  str
    confidence: Confidence
    rate_type:  RateType

@dataclass
class CurrencyOutput:
    status: ToolStatus
    data:   CurrencyData
    meta:   CurrencyMeta

@dataclass
class AttractionCost:
    amount:   Optional[float] = None
    currency: str             = "USD"
    known:    bool            = False   # False != zero; means cost is unknown

@dataclass
class Attraction:
    name:               str
    type:               ActivityType
    estimated_cost:     AttractionCost
    duration_hours:     float
    weather_suitability: WeatherSuitability

@dataclass
class AttractionsData:
    city:        str
    attractions: list = field(default_factory=list)

@dataclass
class AttractionsMeta:
    source:       DataSource
    fallback_used: bool
    timestamp:    str
    confidence:   Confidence

@dataclass
class AttractionsOutput:
    status: ToolStatus
    data:   AttractionsData
    meta:   AttractionsMeta


# ── Layer 3: Validation envelope ─────────────────────────────────

@dataclass
class ValidationEnvelope:
    global_status:        GlobalStatus
    issues:               list
    confidence:           Confidence
    decision_constraints: list
    weather:     Optional[WeatherOutput]     = None
    currency:    Optional[CurrencyOutput]    = None
    attractions: Optional[AttractionsOutput] = None


print("✅ Section 1: Data contracts loaded")
print(f"   ToolStatus values  : {[s.value for s in ToolStatus]}")
print(f"   GlobalStatus values: {[s.value for s in GlobalStatus]}")
print(f"   Supported currencies: {sorted(SUPPORTED_CURRENCIES)}")


---
## Section 2 — Attractions Database

Curated, verified data for 5 cities. No LLM fallback — that would hallucinate attractions.

In [ ]:
import json

ATTRACTIONS_DB = {
    "Kigali": [
        {"name": "Kigali Genocide Memorial", "type": "cultural",
         "estimated_cost": {"amount": 0, "currency": "RWF", "known": True},
         "duration_hours": 2.5, "weather_suitability": "good"},
        {"name": "Inema Arts Center", "type": "indoor",
         "estimated_cost": {"amount": 5000, "currency": "RWF", "known": True},
         "duration_hours": 1.5, "weather_suitability": "good"},
        {"name": "Kimironko Market", "type": "outdoor",
         "estimated_cost": {"amount": 0, "currency": "RWF", "known": True},
         "duration_hours": 1.5, "weather_suitability": "neutral"},
        {"name": "Nyamirambo Women's Center", "type": "cultural",
         "estimated_cost": {"amount": 10000, "currency": "RWF", "known": True},
         "duration_hours": 2.0, "weather_suitability": "good"},
        {"name": "Mount Kigali Hike", "type": "outdoor",
         "estimated_cost": {"amount": 0, "currency": "RWF", "known": True},
         "duration_hours": 3.0, "weather_suitability": "bad"},
        {"name": "Kigali City Tower", "type": "mixed",
         "estimated_cost": {"amount": 0, "currency": "RWF", "known": True},
         "duration_hours": 1.0, "weather_suitability": "good"},
    ],
    "Kampala": [
        {"name": "Uganda Museum", "type": "indoor",
         "estimated_cost": {"amount": 10000, "currency": "UGX", "known": True},
         "duration_hours": 2.0, "weather_suitability": "good"},
        {"name": "Kasubi Tombs", "type": "cultural",
         "estimated_cost": {"amount": 20000, "currency": "UGX", "known": True},
         "duration_hours": 1.5, "weather_suitability": "neutral"},
        {"name": "Owino Market", "type": "outdoor",
         "estimated_cost": {"amount": 0, "currency": "UGX", "known": True},
         "duration_hours": 1.5, "weather_suitability": "bad"},
        {"name": "Namirembe Cathedral", "type": "cultural",
         "estimated_cost": {"amount": 0, "currency": "UGX", "known": True},
         "duration_hours": 1.0, "weather_suitability": "good"},
        {"name": "Gaddafi National Mosque", "type": "cultural",
         "estimated_cost": {"amount": 5000, "currency": "UGX", "known": True},
         "duration_hours": 1.0, "weather_suitability": "good"},
        {"name": "Murchison Falls Day Trip", "type": "outdoor",
         "estimated_cost": {"amount": 40, "currency": "USD", "known": True},
         "duration_hours": 8.0, "weather_suitability": "bad"},
    ],
    "Nairobi": [
        {"name": "Nairobi National Museum", "type": "indoor",
         "estimated_cost": {"amount": 1200, "currency": "KES", "known": True},
         "duration_hours": 2.5, "weather_suitability": "good"},
        {"name": "Nairobi National Park", "type": "outdoor",
         "estimated_cost": {"amount": 60, "currency": "USD", "known": True},
         "duration_hours": 4.0, "weather_suitability": "bad"},
        {"name": "Karen Blixen Museum", "type": "cultural",
         "estimated_cost": {"amount": 1200, "currency": "KES", "known": True},
         "duration_hours": 1.5, "weather_suitability": "good"},
        {"name": "Maasai Market", "type": "outdoor",
         "estimated_cost": {"amount": 0, "currency": "KES", "known": True},
         "duration_hours": 1.5, "weather_suitability": "neutral"},
        {"name": "Giraffe Centre", "type": "outdoor",
         "estimated_cost": {"amount": 1500, "currency": "KES", "known": True},
         "duration_hours": 2.0, "weather_suitability": "bad"},
        {"name": "Bomas of Kenya", "type": "cultural",
         "estimated_cost": {"amount": 1000, "currency": "KES", "known": True},
         "duration_hours": 2.0, "weather_suitability": "good"},
    ],
    "Dar Es Salaam": [
        {"name": "National Museum of Tanzania", "type": "indoor",
         "estimated_cost": {"amount": 10000, "currency": "TZS", "known": True},
         "duration_hours": 2.0, "weather_suitability": "good"},
        {"name": "Kariakoo Market", "type": "outdoor",
         "estimated_cost": {"amount": 0, "currency": "TZS", "known": True},
         "duration_hours": 1.5, "weather_suitability": "bad"},
        {"name": "Village Museum", "type": "cultural",
         "estimated_cost": {"amount": 8000, "currency": "TZS", "known": True},
         "duration_hours": 2.0, "weather_suitability": "neutral"},
        {"name": "Coco Beach", "type": "outdoor",
         "estimated_cost": {"amount": 0, "currency": "TZS", "known": True},
         "duration_hours": 2.5, "weather_suitability": "bad"},
        {"name": "St. Joseph's Cathedral", "type": "cultural",
         "estimated_cost": {"amount": 0, "currency": "TZS", "known": True},
         "duration_hours": 1.0, "weather_suitability": "good"},
    ],
    "Mombasa": [
        {"name": "Fort Jesus", "type": "cultural",
         "estimated_cost": {"amount": 1200, "currency": "KES", "known": True},
         "duration_hours": 2.0, "weather_suitability": "neutral"},
        {"name": "Old Town Mombasa", "type": "outdoor",
         "estimated_cost": {"amount": 0, "currency": "KES", "known": True},
         "duration_hours": 2.0, "weather_suitability": "bad"},
        {"name": "Haller Park", "type": "outdoor",
         "estimated_cost": {"amount": 1000, "currency": "KES", "known": True},
         "duration_hours": 2.5, "weather_suitability": "bad"},
        {"name": "Mama Ngina Waterfront", "type": "outdoor",
         "estimated_cost": {"amount": 0, "currency": "KES", "known": True},
         "duration_hours": 1.5, "weather_suitability": "bad"},
        {"name": "Mombasa Marine Park", "type": "outdoor",
         "estimated_cost": {"amount": 1500, "currency": "KES", "known": True},
         "duration_hours": 3.0, "weather_suitability": "bad"},
    ],
}

SUPPORTED_CITIES = set(ATTRACTIONS_DB.keys())

print(f"✅ Section 2: Database loaded")
print(f"   Cities: {sorted(SUPPORTED_CITIES)}")
print(f"   Total attractions: {sum(len(v) for v in ATTRACTIONS_DB.values())}")


---
## Section 3 — Layer 0: Interpretation

Converts a raw user string → `UserIntent`.

Rule-based extractors handle the common patterns. This layer is the **only** place ambiguity is resolved — everything downstream receives structured, typed data.

| Input | Extracted field |
|-------|-----------------|
| "Kigali", "Nairobi" | `destination` |
| "50 USD", "150000 RWF" | `budget`, `currency` |
| "cheap", "luxury", "mid" | `budget_level` → `budget` |
| "2026-04-01", "today", "next weekend" | `start_date` |
| "3 days", "2-day" | `num_days` |
| "adventure", "leisure" | `trip_type` |


In [ ]:
import re
from datetime import datetime, timedelta

# City aliases
CITY_ALIASES = {
    "dar es salaam": "Dar Es Salaam", "dar": "Dar Es Salaam",
    "nairobi": "Nairobi", "kampala": "Kampala",
    "kigali": "Kigali", "mombasa": "Mombasa",
}

# Budget level → USD/day approximation
BUDGET_LEVEL_MAP = {
    "cheap": ("low",    30.0),
    "budget": ("low",   30.0),
    "low": ("low",      30.0),
    "affordable": ("low", 30.0),
    "mid": ("medium",   80.0),
    "moderate": ("medium", 80.0),
    "medium": ("medium", 80.0),
    "luxury": ("high",  200.0),
    "expensive": ("high", 200.0),
    "high": ("high",    200.0),
    "premium": ("high", 200.0),
}

TRIP_TYPES = {"leisure", "adventure", "cultural", "heritage", "beach", "spiritual", "nature"}
CURRENCY_CODES = {"USD", "EUR", "GBP", "UGX", "KES", "RWF", "TZS"}


def extract_city(text: str) -> Optional[str]:
    t = text.lower()
    # Check aliases first (longer matches first to avoid partial hits)
    for alias in sorted(CITY_ALIASES, key=len, reverse=True):
        if alias in t:
            return CITY_ALIASES[alias]
    # Check supported city names directly
    for city in SUPPORTED_CITIES:
        if city.lower() in t:
            return city
    # Capture unsupported cities via common prepositions (in/to/visit)
    m = re.search(r"(?:in|to|visit|visiting|for)\s+([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)", text)
    if m:
        return m.group(1).strip()
    return None


def extract_budget(text: str) -> tuple:
    """Returns (budget_float, currency_str, budget_level_str, was_inferred)."""
    t = text.upper()

    # Numeric budget with currency: "150000 RWF", "50 USD", "$50"
    for cur in CURRENCY_CODES:
        pattern = rf"(\d[\d,\.]*)\s*{cur}"
        m = re.search(pattern, t)
        if m:
            amount = float(m.group(1).replace(",", ""))
            return amount, cur, None, False
    # Dollar sign: "$50"
    m = re.search(r"\$\s*(\d[\d,\.]*)", t)
    if m:
        return float(m.group(1).replace(",", "")), "USD", None, False

    # Fuzzy budget level
    tl = text.lower()
    for word, (level, approx) in BUDGET_LEVEL_MAP.items():
        if re.search(rf"\b{word}\b", tl):
            return approx, "USD", level, True

    return None, None, None, False


def extract_date(text: str) -> Optional[str]:
    t = text.lower()
    today = datetime.now()

    if "today" in t:
        return today.strftime("%Y-%m-%d")
    if "tomorrow" in t:
        return (today + timedelta(days=1)).strftime("%Y-%m-%d")
    if "next weekend" in t:
        days = (5 - today.weekday()) % 7 or 7
        return (today + timedelta(days=days)).strftime("%Y-%m-%d")
    if "this weekend" in t:
        days = (5 - today.weekday()) % 7
        days = days if days > 0 else 7
        return (today + timedelta(days=days)).strftime("%Y-%m-%d")

    # ISO date: 2026-04-01
    m = re.search(r"\b(\d{4}-\d{2}-\d{2})\b", text)
    if m:
        return m.group(1)

    return None


def extract_days(text: str) -> Optional[int]:
    m = re.search(r"(\d+)[-\s]?day", text.lower())
    if m:
        return int(m.group(1))
    m = re.search(r"for\s+(\d+)\s+days?", text.lower())
    if m:
        return int(m.group(1))
    return None


def extract_trip_type(text: str) -> Optional[str]:
    tl = text.lower()
    for tt in TRIP_TYPES:
        if tt in tl:
            return tt
    return "leisure"  # safe default


def interpret(user_input: str) -> UserIntent:
    """
    Layer 0: convert raw user string → structured UserIntent.
    All ambiguity resolved here — nothing passes through as a raw string.
    """
    budget, currency, budget_level, was_inferred = extract_budget(user_input)

    return UserIntent(
        destination  = extract_city(user_input),
        start_date   = extract_date(user_input),
        num_days     = extract_days(user_input),
        budget       = budget,
        budget_level = budget_level,
        currency     = currency,
        trip_type    = extract_trip_type(user_input),
        budget_was_inferred = was_inferred,
    )


# Quick self-test
_tests = [
    ("Plan a trip in Kigali with 150000 RWF", "Kigali", 150000.0, "RWF"),
    ("I want a cheap trip to Nairobi next weekend", "Nairobi", 30.0, "USD"),
    ("Luxury day in Kampala, today", "Kampala", 200.0, "USD"),
    ("50 USD trip to Mombasa 2026-04-01", "Mombasa", 50.0, "USD"),
]
passed = 0
for txt, exp_city, exp_budget, exp_cur in _tests:
    i = interpret(txt)
    ok = (i.destination == exp_city and i.budget == exp_budget and i.currency == exp_cur)
    if ok:
        passed += 1
    else:
        print(f"  FAIL: {txt!r} → city={i.destination}, budget={i.budget}, currency={i.currency}")

print(f"✅ Section 3: Interpretation layer loaded — {passed}/{len(_tests)} self-tests passed")


---
## Section 4 — Layer 1: Planner

Two phases:
1. **Completeness check** — are all required fields present? If not, return `Plan(requires_clarification=True)` with a specific prompt.
2. **Tool selection** — deterministic rules, returns `Plan(tools, reasoning)`. Every decision is traced.

**Policy decisions (explicit):**
- `destination` and `start_date` are required. `num_days` defaults to 1 if absent.
- Inferred budget (from "cheap"/"luxury") requires user confirmation before proceeding.
- City must be in the supported list — rejection message explains the constraint.


In [ ]:
REQUIRED_FIELDS = ["destination", "start_date"]

def planner(intent: UserIntent) -> Plan:
    """
    Layer 1: completeness check + rule-based tool selection.
    Returns Plan with explicit reasoning trace.
    """
    missing = []

    if not intent.destination:
        missing.append("destination (city name)")

    if not intent.start_date:
        missing.append("date (YYYY-MM-DD, or: today / tomorrow / next weekend)")

    # Budget: inferred budget needs confirmation, no budget at all is missing
    if intent.budget is None:
        missing.append("budget (e.g. 150000 RWF — or a label: cheap / mid / luxury)")

    if not intent.currency:
        missing.append(f"currency (one of: {', '.join(sorted(SUPPORTED_CURRENCIES))})")
    elif intent.currency.upper() not in SUPPORTED_CURRENCIES:
        missing.append(f"currency '{intent.currency}' not supported "
                       f"(use: {', '.join(sorted(SUPPORTED_CURRENCIES))})")

    if missing:
        return Plan(
            requires_clarification=True,
            missing_fields=missing,
            tools=[],
            reasoning="Insufficient parameters — clarification required before tool execution",
        )

    # Inferred budget needs confirmation (not a hard block — just a confirmation step)
    if intent.budget_was_inferred:
        return Plan(
            requires_clarification=True,
            missing_fields=[f"budget_confirm ({intent.budget_level} → ~{intent.budget:.0f} USD/day)"],
            tools=[],
            reasoning=f"Budget inferred from label '{intent.budget_level}' — needs user confirmation",
        )

    # City support check — after confirming all fields present
    dest = intent.destination
    if dest not in SUPPORTED_CITIES:
        return Plan(
            requires_clarification=True,
            missing_fields=[f"destination (supported: {', '.join(sorted(SUPPORTED_CITIES))})"],
            tools=[],
            reasoning=f"'{dest}' not in supported cities — design decision: verified data over broad coverage",
        )

    # Tool selection — deterministic rules, every choice justified
    tools, reasons = [], []

    tools.append("weather")
    reasons.append(
        f"weather: assess activity feasibility for {dest}; "
        "sets decision_constraint (weather_aware vs weather_agnostic)")

    tools.append("attractions")
    reasons.append(
        "attractions: populate itinerary from verified source — no LLM fallback")

    if intent.budget is not None and intent.currency:
        tools.append("currency")
        reasons.append(
            f"currency: user budget ({intent.budget} {intent.currency}) "
            "needs conversion to assess feasibility")

    return Plan(
        requires_clarification=False,
        missing_fields=[],
        tools=tools,
        reasoning=" | ".join(reasons),
    )


def ask_user(plan: Plan, intent: UserIntent) -> str:
    """Generate clarification prompt from plan.missing_fields."""
    if any("budget_confirm" in f for f in plan.missing_fields):
        return (
            f"I interpreted your budget as ~{intent.budget:.0f} USD/day "
            f"({intent.budget_level} level). Date: {intent.start_date}.\n"
            "Shall I proceed? (yes / no, or give a specific number + currency)"
        )
    lines = ["I need a few more details to plan your trip:"]
    for f in plan.missing_fields:
        lines.append(f"  - {f}")
    return "\n".join(lines)


print("✅ Section 4: Planner loaded")


---
## Section 5 — Layer 2: Tool Executor

Three tools. Each returns a **typed schema** — never a raw dict.

This is a deliberate upgrade over the reference architecture which uses `data: dict`. Typed schemas mean:
- Missing fields are explicit (`None`), not absent
- Enums prevent arbitrary string values slipping through
- The validator can inspect fields without string parsing


In [ ]:
import os
import requests
from datetime import datetime, timezone

OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY", "")

# Fallback exchange rates (USD base) — openly declared, not hidden
FALLBACK_RATES = {
    "USD": 1.0, "EUR": 0.92, "GBP": 0.79,
    "UGX": 3720.0, "KES": 129.5, "RWF": 1320.0, "TZS": 2530.0,
}

def _now() -> str:
    return datetime.now(timezone.utc).isoformat()

def _map_owm_condition(desc: str) -> WeatherCondition:
    d = desc.lower()
    if any(k in d for k in ("thunderstorm", "thunder")): return WeatherCondition.THUNDERSTORM
    if any(k in d for k in ("rain", "drizzle", "shower")): return WeatherCondition.RAINY
    if any(k in d for k in ("fog", "mist", "haze")):       return WeatherCondition.FOGGY
    if "overcast" in d:   return WeatherCondition.CLOUDY
    if "cloud" in d:      return WeatherCondition.PARTLY_CLOUDY
    if any(k in d for k in ("clear", "sun")): return WeatherCondition.SUNNY
    return WeatherCondition.UNKNOWN


def get_weather(city: str) -> WeatherOutput:
    """Fetch current weather. Returns UNAVAILABLE cleanly — never invents data."""
    ts = _now()
    agnostic_meta = WeatherMeta(
        source=DataSource.API, timestamp=ts,
        confidence=Confidence.LOW,
        decision_constraint=DecisionConstraint.WEATHER_AGNOSTIC)

    if not OPENWEATHER_API_KEY:
        return WeatherOutput(status=ToolStatus.UNAVAILABLE,
                             data=WeatherData(), meta=agnostic_meta)
    try:
        resp = requests.get(
            "https://api.openweathermap.org/data/2.5/weather",
            params={"q": city, "appid": OPENWEATHER_API_KEY, "units": "metric"},
            timeout=8)
        if resp.status_code == 404:
            return WeatherOutput(status=ToolStatus.ERROR,
                                 data=WeatherData(), meta=agnostic_meta)
        if resp.status_code != 200:
            return WeatherOutput(status=ToolStatus.UNAVAILABLE,
                                 data=WeatherData(), meta=agnostic_meta)
        raw = resp.json()
        cond = _map_owm_condition(raw["weather"][0]["description"])
        precip = (90.0 if "rain" in raw
                  else 30.0 if cond in (WeatherCondition.CLOUDY, WeatherCondition.PARTLY_CLOUDY)
                  else 5.0)
        constraint = (DecisionConstraint.WEATHER_AGNOSTIC
                      if cond == WeatherCondition.UNKNOWN
                      else DecisionConstraint.WEATHER_AWARE)
        return WeatherOutput(
            status=ToolStatus.SUCCESS,
            data=WeatherData(
                temperature_celsius=raw["main"]["temp"], condition=cond,
                precipitation_probability=precip,
                wind_speed_kph=raw["wind"]["speed"] * 3.6),
            meta=WeatherMeta(source=DataSource.API, timestamp=ts,
                             confidence=Confidence.HIGH,
                             decision_constraint=constraint))
    except Exception:
        return WeatherOutput(status=ToolStatus.UNAVAILABLE,
                             data=WeatherData(), meta=agnostic_meta)


def convert_currency(amount: float, from_c: str, to_c: str) -> CurrencyOutput:
    """Convert between currencies. Validates codes before any API call."""
    ts = _now()
    from_c, to_c = from_c.upper().strip(), to_c.upper().strip()

    if from_c not in SUPPORTED_CURRENCIES or to_c not in SUPPORTED_CURRENCIES:
        return CurrencyOutput(
            status=ToolStatus.ERROR,
            data=CurrencyData(amount_input=amount, currency_from=from_c,
                              currency_to=to_c, input_validated=False),
            meta=CurrencyMeta(source=DataSource.API, timestamp=ts,
                              confidence=Confidence.LOW, rate_type=RateType.FIXED))
    try:
        resp = requests.get(
            f"https://api.exchangerate-api.com/v4/latest/{from_c}", timeout=6)
        if resp.status_code == 200:
            rates = resp.json().get("rates", {})
            if to_c in rates:
                rate = rates[to_c]
                return CurrencyOutput(
                    status=ToolStatus.SUCCESS,
                    data=CurrencyData(amount_input=amount, currency_from=from_c,
                                      currency_to=to_c,
                                      converted_amount=round(amount * rate, 2),
                                      exchange_rate=rate, input_validated=True),
                    meta=CurrencyMeta(source=DataSource.API, timestamp=ts,
                                      confidence=Confidence.HIGH, rate_type=RateType.LIVE))
    except Exception:
        pass

    # Fallback to fixed rates — openly labelled
    if from_c in FALLBACK_RATES and to_c in FALLBACK_RATES:
        rate = FALLBACK_RATES[to_c] / FALLBACK_RATES[from_c]
        return CurrencyOutput(
            status=ToolStatus.SUCCESS,
            data=CurrencyData(amount_input=amount, currency_from=from_c,
                              currency_to=to_c,
                              converted_amount=round(amount * rate, 2),
                              exchange_rate=rate, input_validated=True),
            meta=CurrencyMeta(source=DataSource.FALLBACK, timestamp=ts,
                              confidence=Confidence.MEDIUM, rate_type=RateType.FIXED))

    return CurrencyOutput(
        status=ToolStatus.UNAVAILABLE,
        data=CurrencyData(amount_input=amount, currency_from=from_c,
                          currency_to=to_c, input_validated=True),
        meta=CurrencyMeta(source=DataSource.FALLBACK, timestamp=ts,
                          confidence=Confidence.LOW, rate_type=RateType.FIXED))


def get_attractions(city: str) -> AttractionsOutput:
    """Return attractions from curated database. No LLM fallback."""
    ts = _now()
    city_key = city.strip().title()
    if city_key in ATTRACTIONS_DB:
        raw = ATTRACTIONS_DB[city_key]
        acts = [Attraction(
            name=a["name"], type=ActivityType(a["type"]),
            estimated_cost=AttractionCost(
                amount=a["estimated_cost"]["amount"],
                currency=a["estimated_cost"]["currency"],
                known=a["estimated_cost"]["known"]),
            duration_hours=a["duration_hours"],
            weather_suitability=WeatherSuitability(a["weather_suitability"]))
            for a in raw]
        return AttractionsOutput(
            status=ToolStatus.SUCCESS,
            data=AttractionsData(city=city_key, attractions=acts),
            meta=AttractionsMeta(source=DataSource.API, fallback_used=False,
                                 timestamp=ts, confidence=Confidence.HIGH))
    return AttractionsOutput(
        status=ToolStatus.EMPTY,
        data=AttractionsData(city=city_key, attractions=[]),
        meta=AttractionsMeta(source=DataSource.FALLBACK, fallback_used=True,
                             timestamp=ts, confidence=Confidence.LOW))


def execute_tools(plan: Plan, intent: UserIntent) -> dict:
    """
    Layer 2: execute tools from plan.tools.
    Returns dict of tool_name → typed output schema.
    Hard failures caught here — never bubble up to validator as exceptions.
    """
    results = {}
    for tool in plan.tools:
        try:
            if tool == "weather":
                results["weather"] = get_weather(intent.destination)
            elif tool == "attractions":
                results["attractions"] = get_attractions(intent.destination)
            elif tool == "currency":
                results["currency"] = convert_currency(
                    intent.budget, intent.currency, intent.currency)
        except Exception as e:
            # Should never reach here given our typed tools — but catches anything unexpected
            print(f"  [WARN] unexpected exception in tool '{tool}': {e}")
    return results


print("✅ Section 5: Tool executor loaded")
print(f"   Weather API: {'live' if OPENWEATHER_API_KEY else 'degraded mode (no key)'}")


---
## Section 6 — Layer 3: Validator

The "thinking layer" between execution and response.

Responsibilities:
1. Normalise all tool outputs — missing fields become explicit `None`
2. Emit **decision constraints** — behavioral directives the responder must follow
3. Compute **GlobalStatus** — complete / partial / insufficient / terminal
4. Detect **logical inconsistencies** (Type 3 failure) — e.g. rain + outdoor-only plan


In [ ]:
def validate(results: dict) -> ValidationEnvelope:
    """
    Layer 3: cross-tool consistency check and constraint emission.
    Returns ValidationEnvelope — the only thing the Responder is allowed to see.
    """
    issues, constraints = [], []

    weather     = results.get("weather")
    currency    = results.get("currency")
    attractions = results.get("attractions")

    # ── Weather ───────────────────────────────────────────────────
    if weather is None or weather.status in (ToolStatus.UNAVAILABLE, ToolStatus.ERROR):
        issues.append(ValidationIssue.WEATHER_UNAVAILABLE)
        constraints.append(DecisionConstraint.WEATHER_AGNOSTIC)
    else:
        constraints.append(weather.meta.decision_constraint)

    # ── Attractions ───────────────────────────────────────────────
    no_attractions = (
        attractions is None
        or attractions.status in (ToolStatus.UNAVAILABLE, ToolStatus.ERROR, ToolStatus.EMPTY)
        or len(attractions.data.attractions) == 0
    )
    if no_attractions:
        issues.append(ValidationIssue.NO_ATTRACTIONS)

    # ── Currency ──────────────────────────────────────────────────
    if currency is not None:
        if currency.status in (ToolStatus.UNAVAILABLE, ToolStatus.ERROR):
            issues.append(ValidationIssue.CURRENCY_FAILED)
        elif currency.data.converted_amount is None:
            issues.append(ValidationIssue.BUDGET_MISSING)

    # ── Cost completeness ─────────────────────────────────────────
    if attractions and attractions.data.attractions:
        if any(not a.estimated_cost.known for a in attractions.data.attractions):
            issues.append(ValidationIssue.COST_UNKNOWN)

    # ── Type 3 failure: logical inconsistency ─────────────────────
    # Rain + all-outdoor attractions = plan is logically unsound
    if (weather and weather.status == ToolStatus.SUCCESS
            and weather.data.condition in (WeatherCondition.RAINY, WeatherCondition.THUNDERSTORM)
            and attractions and attractions.data.attractions
            and all(a.weather_suitability == WeatherSuitability.BAD
                    for a in attractions.data.attractions)):
        issues.append(ValidationIssue.LOGICAL_CONFLICT)

    # ── Global status ─────────────────────────────────────────────
    weather_gone = ValidationIssue.WEATHER_UNAVAILABLE in issues
    if no_attractions and weather_gone:
        global_status = GlobalStatus.TERMINAL
    elif no_attractions:
        global_status = GlobalStatus.INSUFFICIENT
    elif weather_gone or ValidationIssue.LOGICAL_CONFLICT in issues:
        global_status = GlobalStatus.PARTIAL
    elif ValidationIssue.CURRENCY_FAILED in issues:
        global_status = GlobalStatus.PARTIAL
    else:
        global_status = GlobalStatus.COMPLETE

    # ── Confidence ────────────────────────────────────────────────
    score = 3
    if ValidationIssue.WEATHER_UNAVAILABLE in issues: score -= 1
    if ValidationIssue.NO_ATTRACTIONS in issues:      score -= 2
    if ValidationIssue.CURRENCY_FAILED in issues:     score -= 1
    if attractions and attractions.meta.fallback_used: score -= 1
    confidence = (Confidence.HIGH   if score >= 3 else
                  Confidence.MEDIUM if score >= 2 else
                  Confidence.LOW)

    return ValidationEnvelope(
        global_status=global_status, issues=issues,
        confidence=confidence, decision_constraints=constraints,
        weather=weather, currency=currency, attractions=attractions)


print("✅ Section 6: Validator loaded")


---
## Section 7 — Layer 4: Responder

**Strict rule:** the responder only receives a `ValidationEnvelope`. It never sees the raw user string.

Two guarantees enforced here:
1. **Constraints are BINDING** — `WEATHER_AGNOSTIC` prevents any weather-based characterisation; `LOGICAL_CONFLICT` removes outdoor activities
2. **Confidence drives tone** — `HIGH`=confident, `MEDIUM`=caveated, `LOW`=explicit warning


In [ ]:
def _enforce_constraints(envelope: ValidationEnvelope):
    """
    BINDING enforcement — not advisory.
    LOGICAL_CONFLICT  → outdoor activities removed.
    Returns (filtered_activities, enforcement_notes).
    """
    if not envelope.attractions or not envelope.attractions.data.attractions:
        return [], []
    acts = list(envelope.attractions.data.attractions)
    notes = []
    if ValidationIssue.LOGICAL_CONFLICT in envelope.issues:
        filtered = [a for a in acts if a.weather_suitability != WeatherSuitability.BAD]
        if filtered:
            acts = filtered
            notes.append("Outdoor activities removed due to rain — indoor/cultural only.")
        else:
            notes.append("All activities weather-dependent. Check conditions before visiting.")
    return acts[:4], notes


def _confidence_tone(confidence: Confidence, global_status: GlobalStatus) -> str:
    """Confidence actively changes response language — not just a label."""
    if confidence == Confidence.HIGH and global_status == GlobalStatus.COMPLETE:
        return ""
    if confidence == Confidence.MEDIUM:
        return ("> **Note:** Plan based on partially verified data. "
                "Some information from fallback sources.\n\n")
    return ("> **Warning:** Data confidence LOW. Critical information unavailable. "
            "Treat as a rough starting point — verify before travelling.\n\n")


def terminal_fallback(reason: str, intent: UserIntent) -> str:
    cities = ", ".join(sorted(SUPPORTED_CITIES))
    if reason == "unsupported_city":
        return (f"Can't plan a trip to '{intent.destination}' — not in supported cities.\n\n"
                f"Supported: {cities}\n\n"
                "Options:\n  A) Try a supported city\n  B) Generic day-trip template")
    return (f"Both weather and attraction data unavailable — no plan possible.\n\n"
            f"Options:\n  A) Try a different city (supported: {cities})\n"
            "  B) Generic day-trip template")


def responder(envelope: ValidationEnvelope, intent: UserIntent) -> str:
    """
    Layer 4: build final response from ValidationEnvelope only.
    Raw user string is NOT accessible here — by design.
    """
    city = intent.destination or "Unknown"
    parts = [f"## Day trip — {city}\n"]

    # Confidence-driven tone header
    parts.append(_confidence_tone(envelope.confidence, envelope.global_status))

    # Issue disclosures (always before the plan)
    if envelope.issues:
        disc = ["**Data notes:**"]
        if ValidationIssue.WEATHER_UNAVAILABLE in envelope.issues:
            disc.append("- Weather unavailable — activities not filtered by conditions.")
        if ValidationIssue.CURRENCY_FAILED in envelope.issues:
            disc.append("- Currency conversion failed — costs shown in original currency.")
        if ValidationIssue.LOGICAL_CONFLICT in envelope.issues:
            disc.append("- Rain detected — outdoor activities removed.")
        if ValidationIssue.COST_UNKNOWN in envelope.issues:
            disc.append("- Some costs unconfirmed — marked below.")
        parts.append("\n".join(disc) + "\n")

    # Weather line
    weather_agnostic = DecisionConstraint.WEATHER_AGNOSTIC in envelope.decision_constraints
    w = envelope.weather
    if w and w.status == ToolStatus.SUCCESS:
        temp   = f"{w.data.temperature_celsius:.1f}°C" if w.data.temperature_celsius else "unknown"
        cond   = w.data.condition.value.replace("_", " ")
        precip = f", {w.data.precipitation_probability:.0f}% rain" if w.data.precipitation_probability else ""
        parts.append(f"**Weather:** {cond}, {temp}{precip}\n")
    else:
        parts.append("**Weather:** unavailable — plan is weather-neutral\n")

    # Budget line
    if intent.budget:
        c = envelope.currency
        if c and c.data.converted_amount is not None and c.data.currency_to != c.data.currency_from:
            rate_note = " *(fallback rate)*" if c.meta.rate_type == RateType.FIXED else ""
            parts.append(f"**Budget:** {intent.budget:,.0f} {intent.currency}"
                         f" ≈ {c.data.converted_amount:,.2f} {c.data.currency_to}{rate_note}\n")
        else:
            parts.append(f"**Budget:** {intent.budget:,.0f} {intent.currency}\n")

    if intent.num_days:
        parts.append(f"**Duration:** {intent.num_days} day(s)  |  **Type:** {intent.trip_type or 'leisure'}\n")
    parts.append("")

    # Activities — constraints ENFORCED
    acts, notes = _enforce_constraints(envelope)
    for note in notes:
        parts.append(f"*{note}*\n")

    if acts:
        slots = ["Morning", "Midday", "Afternoon", "Evening"]
        lines = ["**Itinerary:**\n"]
        for i, act in enumerate(acts):
            slot     = slots[i % len(slots)]
            cost_str = (f"{act.estimated_cost.amount:,.0f} {act.estimated_cost.currency}"
                        if act.estimated_cost.known and act.estimated_cost.amount is not None
                        else "cost unknown")
            # CONSTRAINT: weather_agnostic → no weather characterisation allowed
            w_note = ""
            if not weather_agnostic and act.weather_suitability == WeatherSuitability.BAD:
                w_note = " *(weather-dependent — check forecast)*"
            lines.append(
                f"- **{slot}** — {act.name} "
                f"({act.type.value}, ~{act.duration_hours:.0f}h, {cost_str}){w_note}")
        parts.append("\n".join(lines))
    else:
        parts.append("No attractions available for this city.")

    if envelope.attractions:
        m   = envelope.attractions.meta
        src = "curated database" if not m.fallback_used else "fallback"
        parts.append(f"\n*Source: {src} | confidence: {m.confidence.value} | {m.timestamp[:10]}*")

    return "\n".join(parts)


print("✅ Section 7: Responder loaded")


---
## Section 8 — The Agent Loop

The canonical `agent_loop(user_input: str)` ties all five layers together.
Input is a natural language string. Every intermediate state is logged.


In [ ]:
def agent_loop(user_input: str) -> str:
    """
    Full agent loop — string in, string out.

    [0] interpret  → UserIntent
    [1] planner    → Plan (with reasoning trace)
    [2] execute    → tool results (typed schemas)
    [3] validate   → ValidationEnvelope
    [4] respond    → final string (envelope only, no raw input)
    """
    print(f"[INPUT] {user_input!r}\n")

    # [0] Interpretation
    intent = interpret(user_input)
    print(f"[INTENT] destination={intent.destination}, budget={intent.budget} "
          f"{intent.currency}, date={intent.start_date}, days={intent.num_days}, "
          f"trip_type={intent.trip_type}")
    if intent.budget_was_inferred:
        print(f"[INTENT] budget_level='{intent.budget_level}' (inferred — needs confirmation)")
    print()

    # [1] Planning
    plan = planner(intent)
    print(f"[PLAN] requires_clarification={plan.requires_clarification}")
    print(f"[PLAN] tools={plan.tools}")
    print(f"[PLAN] reasoning={plan.reasoning}")
    print()

    if plan.requires_clarification:
        response = ask_user(plan, intent)
        # Check for terminal city case
        if intent.destination and intent.destination not in SUPPORTED_CITIES and not any(
                "budget" in f or "date" in f or "currency" in f
                for f in plan.missing_fields):
            return terminal_fallback("unsupported_city", intent)
        return response

    # [2] Tool execution
    results = execute_tools(plan, intent)
    for name, result in results.items():
        print(f"[TOOL:{name}] status={result.status.value}")
    print()

    # [3] Validation
    envelope = validate(results)
    print(f"[VALIDATION] global_status={envelope.global_status.value}, "
          f"confidence={envelope.confidence.value}")
    print(f"[VALIDATION] issues={[i.value for i in envelope.issues]}")
    print(f"[VALIDATION] constraints={[c.value for c in envelope.decision_constraints]}")
    print()

    if envelope.global_status == GlobalStatus.TERMINAL:
        return terminal_fallback("no_data", intent)

    # [4] Response
    return responder(envelope, intent)


print("✅ Section 8: Agent loop ready")
print()
print("Usage: agent_loop(\"Plan a trip in Kigali, 150000 RWF, today\")")


---
## Section 9 — Evaluation Transcripts

Eight test scenarios required by the rubric — input is now a **natural language string** throughout.


In [ ]:
print("=" * 65)
print("TRANSCRIPT 1 — Multi-tool happy path")
print(f"User: Plan a one-day trip to Kigali with a budget of 150000 RWF today")
print("=" * 65)
result = agent_loop("Plan a one-day trip to Kigali with a budget of 150000 RWF today")
print(result)

In [ ]:
print("=" * 65)
print("TRANSCRIPT 2 — Clarification: missing fields")
print(f"User: Plan me a trip in Kampala")
print("=" * 65)
result = agent_loop("Plan me a trip in Kampala")
print(result)

In [ ]:
print("=" * 65)
print("TRANSCRIPT 3 — Unsupported city → terminal fallback")
print(f"User: I want to visit Paris with a budget of 200 USD today")
print("=" * 65)
result = agent_loop("I want to visit Paris with a budget of 200 USD today")
print(result)

In [ ]:
print("=" * 65)
print("TRANSCRIPT 4 — Weather tool failure → graceful degradation")
print(f"User: Plan a day trip to Nairobi, 50 USD, 2026-04-01")
print("=" * 65)
result = agent_loop("Plan a day trip to Nairobi, 50 USD, 2026-04-01")
print(result)

In [ ]:
print("=" * 65)
print("TRANSCRIPT 5 — Fuzzy budget → confirmation required")
print(f"User: Plan a cheap trip to Nairobi next weekend")
print("=" * 65)
result = agent_loop("Plan a cheap trip to Nairobi next weekend")
print(result)

In [ ]:
print("=" * 65)
print("TRANSCRIPT 6 — Invalid currency → phase 1 rejection")
print(f"User: Plan a trip to Mombasa, 500 DOLLARS, today")
print("=" * 65)
result = agent_loop("Plan a trip to Mombasa, 500 DOLLARS, today")
print(result)

In [ ]:
print("=" * 65)
print("TRANSCRIPT 7 — Rain + outdoor plan (logical conflict, simulated)")
print("User: Plan a day in Mombasa, 200 USD, today")
print("=" * 65)

# Simulate rainy weather to demonstrate Type 3 failure handling
intent7 = interpret("Plan a day in Mombasa, 200 USD, today")
plan7   = planner(intent7)
results7 = execute_tools(plan7, intent7)

# Override weather with simulated rain
results7["weather"] = WeatherOutput(
    status=ToolStatus.SUCCESS,
    data=WeatherData(temperature_celsius=27.0, condition=WeatherCondition.RAINY,
                     precipitation_probability=85.0, wind_speed_kph=20.0),
    meta=WeatherMeta(source=DataSource.API, timestamp=_now(), confidence=Confidence.HIGH,
                     decision_constraint=DecisionConstraint.WEATHER_AWARE))

envelope7 = validate(results7)
print(f"[VALIDATION] issues={[i.value for i in envelope7.issues]}")
print()
print(responder(envelope7, intent7))


In [ ]:
print("=" * 65)
print("TRANSCRIPT 8 — LOW confidence response tone")
print("(Simulated: weather unavailable + fallback attraction source)")
print("=" * 65)

intent8 = interpret("Plan a trip in Kigali, 100 USD, today")
low_att = AttractionsOutput(
    status=ToolStatus.SUCCESS,
    data=AttractionsData(city="Kigali",
                         attractions=get_attractions("Kigali").data.attractions),
    meta=AttractionsMeta(source=DataSource.FALLBACK, fallback_used=True,
                         timestamp=_now(), confidence=Confidence.LOW))

env8 = validate({"attractions": low_att})
print(f"[VALIDATION] confidence={env8.confidence.value}, status={env8.global_status.value}")
print()
print(responder(env8, intent8))


---
## Section 10 — Threat Model

| Threat | Mitigation |
|--------|------------|
| LLM hallucinating weather/attractions | Responder sees `ValidationEnvelope` only — never raw user string |
| Fuzzy budget entering plan silently | `budget_was_inferred=True` triggers confirmation before any tool is called |
| Currency injection (`100 USD; drop table`) | Codes validated against closed enum before API call |
| Stale exchange rates | `rate_type=FIXED` openly labelled — responder discloses in output |
| Invented attractions | No LLM fallback — `EMPTY` status returned honestly |
| Constraint declared but not enforced | `_enforce_constraints()` is BINDING — outdoor activities removed, not warned |
| Low confidence producing confident output | `_confidence_tone()` changes language, not just adds a label |
| Planning decisions unjustifiable | `Plan.reasoning` traces every tool selection explicitly |
| API key exposure | Keys loaded from env vars only — never hardcoded |
| Brittle string parsing | `extract_*` functions fail explicitly — missing fields surface in `plan.missing_fields` |


---
## Section 11 — Design Trade-offs

This system prioritises **reliability and epistemic control** over conversational flexibility.

| Trade-off | Decision | Justification |
|-----------|----------|---------------|
| 5 cities only | Controlled scope | All attraction data is verified — coverage sacrificed for reliability |
| Typed schemas over raw dicts | Stricter than reference architecture | Missing fields are explicit; enums prevent invalid values |
| Rule-based tool selector | Deterministic | Reproducible evaluation transcripts; no LLM non-determinism in routing |
| No LLM attractions fallback | `EMPTY` status | Hallucinated attractions break the grounding guarantee entirely |
| Fuzzy budget with confirmation | Not hard-rejected | Improves UX while preserving epistemic integrity — user confirms before plan proceeds |
| `Plan.reasoning` logged | Explicit trace | Every tool decision is auditable — "why did you call weather?" has a concrete answer |
| Degraded mode without API key | Still functional | System fails loudly on weather, not silently |


---
## Section 12 — Module-Based Agent Demo

The cells above use the self-contained notebook implementation.
This section demos the same scenarios using the **production module code** in `core/`, `agent/`, and `tools/`.
Both implementations follow the same architecture — this shows they produce consistent results.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from core.agent import plan_trip  # loads .env automatically

def show(result):
    status = result.get("status")
    print(f"Status : {status}")
    if status == "needs_clarification":
        print(f"Agent  : {result['message']}")
        print(f"Missing: {result['missing_fields']}")
    elif status == "error":
        print(f"Error  : {result.get('error')}")
    else:
        print(f"City   : {result['city']}")
        print(f"Tools  : {list(result['tool_outputs'].keys())}")
        weather_src = result["tool_outputs"].get("weather", {}).get("source", "n/a")
        print(f"Weather source: {weather_src}")
        if result.get("errors"):
            print(f"Errors : {[e['tool'] for e in result['errors']]}")
        print()
        print(result.get("itinerary", ""))

print("Module agent ready.")

### M1 — Happy path: Kigali + UGX budget

In [ ]:
show(plan_trip({
    "city": "Kigali",
    "budget": 150000,
    "from_currency": "UGX",
    "to_currency": "USD"
}))

### M2 — Clarification: missing city

In [ ]:
show(plan_trip({"budget": 50}))

### M3 — Tool failure: unsupported city

In [ ]:
show(plan_trip({"city": "Tokyo"}))

### M4 — New cities (Nairobi, Kampala, London)

In [ ]:
for city in ["Nairobi", "Kampala", "London"]:
    r = plan_trip({"city": city})
    tools = list(r.get("tool_outputs", {}).keys())
    print(f"{city}: status={r['status']}, tools={tools}")

### M5 — LLM natural language input (requires ANTHROPIC_API_KEY)

In [ ]:
# Set ANTHROPIC_API_KEY in .env to activate LLM planner + responder
show(plan_trip({
    "message": "Plan a one-day trip in Kigali with a budget of 150,000 UGX, show costs in USD."
}))